# Milestone 2: Get Started

## Task 1: Download the Pinterest infrastructure
Here we set up the RDS database on locally on [DBeaver](https://dbeaver.io/) using MySQL, alternatively you can set this up on aws to run it on the cloud. We'll store the database credentials in a `local_db_creds.yaml` or `aws_db_creds.yaml` file depending on which environment you are using.

## Task 2: Sign in to the AWS console
Here we used our AWS IAM account to sign in to the AWS console. The working region is set to `eu-west-1` throughout this project.

## Configure our own MySQL database to host our data
We either set up our own local database or use a cloud database service like AWS RDS. Let's discuss both

### Setting up our own database locally

1. For macOS, we can use Homebrew to install MySQL:
    ```
    # Install via Homebrew
    brew install mysql

    # Start MySQL service
    brew services start mysql

    # Secure installation (set root password)
    mysql_secure_installation
    ```
2. Download (DBeaver Community Edition)[https://dbeaver.io]. Install and launch DBeaver.
3. Connect DBeaver to MySQL
    Open DBeaver → **Database** → **New Database Connection**.

    Select **MySQL** → Click **Next**.

    Configure the connection:
    
    **Host**: localhost

    **Port**: 3306

    **Username**: root

    **Password**: Enter the root password you set during MySQL installation.

    **Database**: Leave empty (create a new database later).

    Test the connection → Click **Finish**.

4. Create a New Database

    In DBeaver: Right-click your MySQL connection → **Create** → **Database**.
    Name the database (e.g., `pinterest_data_db`) → Click **OK**.

5. Import the SQL File

    Open the `pinterest_data_db` database in DBeaver.
    Right-click `pinterest_data_db` → Tools → Execute Script (or press Ctrl+Shift+X).
    Select your `pinterest_data_db.sql` file → Click **Start**.
    Wait for the script to execute. Check the Log tab for errors.

6. Verify the Import

    Expand the `pinterest_data_db` database → **Tables**.
    Right-click a table → **View Data** to confirm data exists.

**Other useful commands:**

To connect to mysql database run:
```
mysql -u root
```
To change the password of the root user run:
```
mysql -u root -p
ALTER USER 'root'@'localhost' IDENTIFIED BY 'new_password';
```
Please note that the username and password you set for the MySql database should be stored in `local_db_creds.yaml` for connnecting to the database later when running the `user_posting_emulation.py` file.

To start/stop/restart the mysql service run:
```
brew services start mysql
brew services stop mysql
brew services restart mysql
```

### TODO: Setting up our own database on AWS RDS

## Debugging MySQL Connection in DBeaver

- The error **"Public Key Retrieval is not allowed"** typically occurs when connecting to MySQL 8.0+ with certain security configurations. Here's how to fix it in DBeaver:

    Step 1: Edit Your MySQL Connection in DBeaver

    In DBeaver, right-click your MySQL connection → **Edit Connection**.
    Go to the **Connection Settings** tab.

    Step 2: Allow Public Key Retrieval

    Under the **Driver Properties** tab:

    Search for the property `allowPublicKeyRetrieval`.
    Set its value to `TRUE`.
    This bypasses the public key retrieval restriction for authentication.

# Milestone 3: Batch Processing: Configure the EC2 Kafka client

## Task 1: Create a .pem key file locally

If You Still Have the Original `.pem` Key when the EC2 was created, Use the existing key to SSH into the instance:
```
ssh -i "~/.ssh/mykeypair.pem" ubuntu@<public-ec2-ip>
```

If You’ve Lost the Original Key, Create a New Key and Add the New Public Key to the Instance:
1. On your local machine, create a new key file:
   ```
   ssh-keygen -y -f mykeypair.pem
   ```
2. Copy/Extract the Public Key from your new `.pem` file, then connect to the EC2 Instance via EC2 Instance Connect and Edit the `~/.ssh/authorized_keys` file:
    ```
    sudo nano ~/.ssh/authorized_keys
    ```
    Replace the existing public key with the new one (or add it as a new line if you want to keep both keys), save and exit `(Ctrl+O, Ctrl+X)`.

## Task 2: Connect to the EC2 instance

1. Here we use the `pinterest-ec2` instance on AWS for this project
2. [`Remote - SSH`](https://marketplace.visualstudio.com/items?itemName=ms-vscode-remote.remote-ssh) extension bug in vscode: 
    
    Shortly after starting up the EC2 instance, you may experience the issue of excessive CPU usage by the 'rg' or/and the 'node' process in VSCode (run `top` to monitor CPU usage on the EC2 instance). This can be resolved by first killing the 'rg' or/and the 'node' processes (`kill -9 <rg_PID> <node_PID>`), then setting `"search.followSymlinks"` to false in VSCode Settings: `Command + , (macOS shorcut)`/`Ctrl + , (Windows shorcut)` -> `Search Settings (Settings.json)` -> add this line `"search.followSymlinks": false`. After that, restart the EC2 instance and ssh to it. See [GitHub Issue #98594](https://github.com/microsoft/vscode/issues/98594) for more information.

3. Setting up Elastic IP on AWS: To retain the same public IP address and DNS name after restarts, use AWS Elastic IP (EIP): Go to `EC2 Dashboard` → `Elastic IPs` → `Allocate Elastic IP address`. Then, select the Elastic IP → `Action` → `Associate Elastic IP address` → Choose your EC2 instance and click `Associate` .
4. Final config in `~/.ssh/config`:
    ```
    ###########################################
    ########### pinterest-ec2 Login ###########
    ###########################################
    Host aws-pinterest-ec2
        HostName <your_aws_ec2_elastic_ip>
        User ubuntu
        IdentityFile ~/.ssh/mykeypair.pem
    ```

## Task 3: Create Kafka Topics on EC2

Find your UserId/Account ID using the AWS CLI:
```
aws sts get-caller-identity
```
Under the "Account" section, you will find your Account ID. Here are our three Kafka topics to create:
```
<account_ID>.pin for the Pinterest posts data 
<account_ID>.geo for the post geolocation data
<account_ID>.user for the post user data

```
(where account_ID = your_UserId)

<!-- topics=808492447622.pin,808492447622.geo,808492447622.user -->

Create a Kafka topic:
```
kafka-topics --create \
  --bootstrap-server localhost:9092 \
  --replication-factor 1 \
  --partitions 3 \
  --topic <kafka-test-topic>
```
Delete a Kafka topic:
```
kafka-topics --delete \
  --bootstrap-server localhost:9092 \
  --topic <kafka-test-topic>
```
List all the Kafka topics that are currently available on the Kafka broker running at localhost:9092
```
kafka-topics --list \
  --bootstrap-server localhost:9092
```

Verify with kafka-console-consumer on EC2 instance:
```
kafka-console-consumer --bootstrap-server localhost:9092 --topic <account_ID>.pin  --from-beginning
```

Delete a topic:
```
kafka-topics --bootstrap-server localhost:9092 --delete --topic <account_ID>.pin
```
Confirm a topic exist and has data:
```
kafka-topics --describe --bootstrap-server localhost:9092 --topic <account_ID>.pin
```
Restart the Connector Service and check logs
```
sudo systemctl daemon-reload
sudo systemctl restart kafka-connect.service
journalctl -u kafka-connect.service -f
# curl -X POST http://localhost:8083/connectors/s3-sink/restart
```
Monitor the connector status again with:
```
curl http://localhost:8083/connectors/s3-sink/status
```
Test if the EC2 instance can access the S3 bucket using the AWS CLI:
```
aws s3 ls s3://<your_bucket_name>
```


# Milestone 4: Batch Processing: Configuring an API in API Gateway

## Task 1: Build a Kafka REST proxy integration method for the API

Follow the instructions in the `2. Integrating API Gateway with Kafka.ipynb`
notebook to complete this part.
Keys things to remember:
- *Security Groups: Ensure your EC2 security group allows inbound traffic on:*
    ```
    2181 (ZooKeeper)
    9092 (Kafka Server)
    8081 (Schema Registry)
    8082 (Kafka REST Proxy)
    8083 (Kafka Connect)
    ```
- *IAM Role: Verify the EC2 instance has permissions to access S3 (if using Kafka Connect S3 sink)*: Here `EC2-S3FullAccess` was used for the IAM Role.
  

## Edite java properties files:

The file `/home/ubuntu/kafka/etc/kafka/s3-sink.properties` is a configuration file for the Confluent S3 Sink Connector, which is part of Kafka Connect. This connector is used to export data from Apache Kafka topics to Amazon S3 in a structured format (e.g., JSON, Avro). Here we need to configure the following properties:
```
topics=<account_ID>.pin,<account_ID>.geo,<account_ID>.user 
s3.region=<your_s3_region> #Set your AWS region
s3.bucket.name=<your_pinterest_confluent_kafka_connect_s3> #Set your S3 bucket name
format.class=io.confluent.connect.s3.format.json.JsonFormat
```

For more information, check [Amazon S3 Source Connector for Confluent Cloud](https://docs.confluent.io/cloud/current/connectors/cc-s3-source.html#using-the-confluent-cli) for using the connector

Here we only need to configure three properties files: `s3-sink.properties`, `server.properties` and `kafka-rest.properties`. And below are some of the important properties that we need to configure:

- For Socket Server Settings In `server.properties`:
    ```
    listeners=PLAINTEXT://0.0.0.0:9092
    advertised.listeners=PLAINTEXT://localhost:9092
    ```
- For S3 Sink Connector Settings In `s3-sink.properties`:
    ```
    # Replace PLACEHOLDER with your UserId
    topics=PLACEHOLDER.pin,PLACEHOLDER.geo,PLACEHOLDER.user
    # Choose the correct region for your S3 bucket
    s3.region=eu-west-1
    # Put the name of your S3 bucket here
    s3.bucket.name=pinterest-confluent-kafka-connect-s3
    format.class=io.confluent.connect.s3.format.json.JsonFormat
    ```
- In `kafka-rest.properties`
    ```
    schema.registry.url=http://localhost:8081
    zookeeper.connect=http://localhost:2181
    bootstrap.servers=PLAINTEXT://localhost:9092
    ```
As a side note, make sure `zookeeper.properties` is correctly configured as well.

If we run `sudo systemctl status kafka-connect.service` then we can see that we're running the `connect-standalone` bin command, using the `connect-standalone.properties` and `s3-sink.properties` files for the configured properties.
```
[Unit]
Description=Apache Kafka Connect - distributed
Documentation=http://docs.confluent.io/
After=network.target kafka-server.service

[Service]
Type=simple
ExecStart=/home/ubuntu/kafka/bin/connect-standalone /home/ubuntu/kafka/etc/kafka/connect-standalone.properties /home/ubuntu/kafka/etc/kafka/s3-sink.properties
TimeoutStopSec=180
Restart=no

Environment="KAFKA_OPTS=-Dcom.amazonaws.sdk.debug=all"

[Install]
WantedBy=multi-user.target
```

## Run Kafka related services

After you've edited the Java properties files, you can run the following command to restart the services for the changes to take effect.:
```
sudo systemctl stop zookeeper.service
sudo systemctl stop kafka-server.service
sudo systemctl stop kafka-rest.service
sudo systemctl stop kafka-connect.service
sudo systemctl stop schema-registry.service

sudo systemctl start zookeeper.service
sudo systemctl start kafka-server.service
sudo systemctl start kafka-rest.service
sudo systemctl start kafka-connect.service
sudo systemctl start schema-registry.service
```
Alternatively albeit less reliable:
```
sudo systemctl restart zookeeper.service
sudo systemctl restart kafka-server.service
sudo systemctl restart kafka-rest.service
sudo systemctl restart kafka-connect.service
sudo systemctl restart schema-registry.service
```
To view the status of the restarted services:
```
sudo systemctl status zookeeper.service
sudo systemctl status kafka-server.service
sudo systemctl status kafka-rest.service
sudo systemctl status kafka-connect.service
sudo systemctl status schema-registry.service
```

Other useful commands systemctl commands:

List all active systemd units related to Zookeeper, Kafka, Kafka Server, and Kafka Connect by filtering the output of systemctl list-units for each respective service:
```
systemctl list-units  | grep zookeeper
systemctl list-units  | grep kafka
systemctl list-units  | grep kafka-server
systemctl list-units  | grep kafka-connect
```


## Task 2: Send data to API Gateway

- Expected output from running `curl http://localhost:8083/connectors/s3-sink/status`:
    ```
    ubuntu@ip-172-31-36-151:~$ curl http://localhost:8083/connectors/s3-sink/status
    {"name":"s3-sink","connector":{"state":"RUNNING","worker_id":"172.31.36.151:8083"},"tasks":[{"id":0,"state":"RUNNING","worker_id":"172.31.36.151:8083"}],"type":"sink"}
    ```
- Expected output from running `journalctl -u kafka-rest.service -f`:
    ```
    ubuntu@ip-172-31-36-151:~$ journalctl -u kafka-rest.service -f
    Mar 10 17:38:00 ip-172-31-36-151 kafka-rest-start[630]: [2025-03-10 17:38:00,994] INFO Started o.e.j.s.ServletContextHandler@595f4da5{/,null,AVAILABLE} (org.eclipse.jetty.server.handler.ContextHandler:921)
    Mar 10 17:38:01 ip-172-31-36-151 kafka-rest-start[630]: [2025-03-10 17:38:01,257] INFO Started o.e.j.s.ServletContextHandler@42561fba{/ws,null,AVAILABLE} (org.eclipse.jetty.server.handler.ContextHandler:921)
    Mar 10 17:38:01 ip-172-31-36-151 kafka-rest-start[630]: [2025-03-10 17:38:01,439] INFO Started NetworkTrafficServerConnector@65b104b9{HTTP/1.1, (http/1.1, h2c)}{0.0.0.0:8082} (org.eclipse.jetty.server.AbstractConnector:333)
    Mar 10 17:38:01 ip-172-31-36-151 kafka-rest-start[630]: [2025-03-10 17:38:01,441] INFO Started @31699ms (org.eclipse.jetty.server.Server:415)
    Mar 10 17:38:01 ip-172-31-36-151 kafka-rest-start[630]: [2025-03-10 17:38:01,442] INFO Server started, listening for requests... (io.confluent.kafkarest.KafkaRestMain:48)
    Mar 11 01:52:31 ip-172-31-36-151 kafka-rest-start[630]: [2025-03-11 01:52:31,112] INFO 159.89.54.154 - - [11/Mar/2025:01:52:30 +0000] "GET / HTTP/1.1" 200 22 "-" "Mozilla/5.0 (compatible)" 392 - (io.confluent.rest-utils.requests:62)
    Mar 11 01:52:31 ip-172-31-36-151 kafka-rest-start[630]: [2025-03-11 01:52:31,428] INFO 159.89.54.154 - - [11/Mar/2025:01:52:31 +0000] "GET /favicon.ico HTTP/1.1" 404 5133 "http://18.200.92.19:8082/" "Mozilla/5.0 (compatible)" 83 - (io.confluent.rest-utils.requests:62)
    Mar 11 05:15:03 ip-172-31-36-151 kafka-rest-start[630]: [2025-03-11 05:15:03,177] INFO 162.142.125.126 - - [11/Mar/2025:05:15:03 +0000] "GET / HTTP/1.1" 200 22 "-" "Mozilla/5.0 (compatible; CensysInspect/1.1; +https://about.censys.io/)" 5 - (io.confluent.rest-utils.requests:62)
    Mar 11 05:15:05 ip-172-31-36-151 kafka-rest-start[630]: [2025-03-11 05:15:05,186] INFO 162.142.125.126 - - [11/Mar/2025:05:15:05 +0000] "GET /favicon.ico HTTP/1.1" 404 5133 "-" "Mozilla/5.0 (compatible; CensysInspect/1.1; +https://about.censys.io/)" 4 - (io.confluent.rest-utils.requests:62)
    Mar 11 05:15:09 ip-172-31-36-151 kafka-rest-start[630]: [2025-03-11 05:15:09,086] INFO 162.142.125.126 - - [11/Mar/2025:05:15:09 +0000] "GET /favicon.ico HTTP/1.1" 404 5133 "-" "Mozilla/5.0 (compatible; CensysInspect/1.1; +https://about.censys.io/)" 8 - (io.confluent.rest-utils.requests:62)
    ```

### Debugging

Issues I had when running this part of the project:
1. The specified bucket is not valid.
    ```
    • ubuntu@ip-172-31-36-151:~$ curl http://localhost: 8083/connectors/s3-sink/status
    {"name": "s3-sink", "connector" : {"state" : "RUNNING"
    ', "worker_id": "172.31.36.151:8083"}, "tasks": [{"id" :0, "state": "FAILED", "worker_id": "172.31
    -36.151:8083"
    ',"trace": "org-apache.kafka. connect.errors.ConnectException: com.amazonaws.services.s3.model.AmazonS3Exception: The specifie
    d bucket is not valid. (Service: Amazon S3; Status Code: 400; Error Code: InvalidBucketName; Request ID: 3EPCDZ7G5VXEH0J7; S3 Extended R equest ID: BdpY/Y/9wUWQ1qMEexyGqVtMuMzgHb0EMdy2sV/7a2zl0GfjJuFYbuNr26Ch72G0YaTd+E1WlYF59KyVPicz2iNKDkd0VRTRQ3kn40o0N2Q=; Proxy: nutt),
    3 Extended Request ID: BdpY/Y/9wUWQ1qMЕexyGqVtMuMzgHb0EMdy2sV/7a2zl0GfjJuFYbuNr26Ch72G0YaTd+E1WlYF59KyVPicz2iNKDkd0VRTRQ3kn40o0N2Q=\n\ta
    t io.confluent.connect.s3.S3SinkTask.start(S3SinkTask,java:142)\n\tat org.apache.kafka.connect.runtime.WorkerSinkTask.initializeAndStart
    (WorkerSinkTask.java:333)\n\tat org-apache.kafka. connect. runtime.WorkerTask.doRun(WorkerTask.java:227)\n\tat org-apache.kafka. connect.ru
    ntime.WorkerTask. run (WorkerTask.java: 284)\n\tat
    org.apache.kafka.connect.runtime.isolation.Plugins.lambda$withClassLoader$7(Plugins-java
    :339)\n\tat java.base/java.util.concurrent.Executors$RunnableAdapter.call(Executors.java:515)\n\tat java.base/java.util.concurrent.Futur
    eTask. run (FutureTask.java:264)\n\tat java.base/java.util.concurrent.ThreadPoolExecutor. runWorker(ThreadPoolExecutor-java:1128)\n\tat jav a.base/java.util.concurrent.ThreadPoolExecutor$Worker. run(ThreadPoolExecutor.java:628)\n\tat java.base/java.lang.Thread. run(Thread-java:
    829) \nCaused by: com.amazonaws.services.s3.model.AmazonS3Exception: The specified bucket is not valid. (Service: Amazon S3; Status Code:
    400; Error Code: InvalidBucketName; Request ID: 3EPCDZ7G5VXEH0J7; S3 Extended Request ID: BdpY/Y/9wUWQ1qMЕexyGqVtMuMzgHb0EMdy2sV/7a2z10
    GfjJuFYbuNr26Ch72G0YaTd+E1W1YF59KyVPicz2iNKDkd0VRTRQ3kn40o0N2Q=; Proxy: null), S3 Extended Request ID: BdpY/Y/9wUWQ1qMEexyGqVtMuMzgHb0EM
    dy2sV/7a2z10GfjJuFYbuNr26Ch72G0YaTd+E1WlYF59KyVPicz2iNKDkd0VRTRQ3kn40o0N2Q=\n\tat com.amazonaws.http.AmazonHttpClient$RequestExecutor.ha
    ndleErrorResponse(AmazonHttpClient.java: 1879)\n\tat com.amazonaws.http.AmazonHttpClient$RequestExecutor.handleServiceErrorResponse (Amazo
    nHttpClient. java: 1418)\n\tat com.amazonaws.http.AmazonHttpClient$RequestExecutor-executeOneRequest(AmazonHttpClient.java:1387)\n\tat com
    .amazonaws.http.AmazonHttpClient$RequestExecutor.executeHelper(AmazonHttpClient.java:1157)\n\tat com.amazonaws.http.AmazonHttpClient$Req
    uestExecutor.doExecute(AmazonHttpClient.java:814)\n\tat com.amazonaws.http.AmazonHttpClient$RequestExecutor.executeWithTimer(AmazonHttpC
    lient. java:781)\n\tat com.amazonaws.http.AmazonHttpClient$RequestExecutor.execute(AmazonHttpClient.java:755)\n\tat com.amazonaws.http.Am
    azonHttpClient$RequestExecutor.access$500(AmazonHttpClient.java:715)\n\tat com.amazonaws.http.AmazonHttpClient$RequestExecutionBuilderIm
    pl.execute(AmazonHttpClient.java:697)\n\tat com.amazonaws.http.AmazonHttpClient.execute(AmazonHttpClient.java:561)\n\tat com.amazonaws.h
    ttp.AmazonHttpClient.execute(AmazonHttpClient.java:541)\n\tat com.amazonaws.services.s3.AmazonS3Client.invoke(AmazonS3Client.java:5520)\
    n\tat com.amazonaws.services.s3.AmazonS3Client. invoke(AmazonS3Client.java:5467)\n\tat com.amazonaws.services.s3.AmazonS3Client.getAcl(Am
    azonS3Client. java: 4113)\n\tat com.amazonaws.services.s3.AmazonS3Client.getBucketAcl(AmazonS3Client.java:1308)\n\tat com.amazonaws.servic
    es. s3.AmazonS3Client.getBucketAcl(AmazonS3Client.java:1298)\n\tat com.amazonaws.services.s3.AmazonS3Client.doesBucketExistV2(AmazonS3Cli
    ent. java: 1436)\n\tat io.confluent.connect.s3.storage.S3Storage.bucketExists(S3Storage.java:186)\n\tat io.confluent.connect.s3.53SinkTask
    ```
    The reason for the error was because in java property files, a logical line cannot be followed by a comment line in the line. See example below from the `s3-sink.properties` file:
    ```
    s3.bucket.name=pinterest-confluent-kafka-connect-s3 # Replace this with your s3 bucket name
    ```
    The above line will not work as the element for the key (`s3.bucket.name`) will be treated as a whole (`pinterest-confluent-kafka-connect-s3 # Replace this with your s3 bucket name`) which is not what we want. See [java Properties Class -> load()](https://docs.oracle.com/javase/7/docs/api/java/util/Properties.html) for more info. The correct syntax should be:
    ```
    # Replace this with your s3 bucket name
    s3.bucket.name=pinterest-confluent-kafka-connect-s3
    ```
2. For this line `log4j.appender.connectAppender.layout=org.apache.log4j.PatternLayout` in file `connect-log4j.properties`, the `l` in `log4j.appender.connectAppender.layout` was origianlly written as a uppercase `L` instead of lowercase `l` which caused the error.

Other useful commands:

To view the logs for the `kafka-connect.service` in reverse chronological order:
```
journalctl -u kafka-connect.service -r
```
Manages cron jobs in edit mode, which are scheduled tasks that run automatically at specified intervals.
```
crontab -e
```


# Milestone 5: Batch Processing: Databricks

## Configure Databricks to allow permissions to read in data from S3.

Go to `Pinterest Data Engineering Project` workspace homepage on Databricks. Click on `+ New` on top left -> `Add or upload data` -> `Create table from Amazon S3` -> Click on existing external s3 location -> Click on `+`(Create new) -> Under `Create a new external location` popup -> slect `AWS Quickstart (Recommended)` -> `Next` -> Bucket Name: `s3://pinterest-confluent-kafka-connect-s3/` -> Personal Access Token: Click on `Generate new token` -> `Launch in Quickstart` -> copy the generated token and paste into the 'Databricks Personal Access Token' field of the CloudFormation template -> `Create stack`. Then wait for `CREATE_COMPLETE` status to show up in CloudFormation > Stacks > databricks-s3-ingest-xxxxx. You can click on template while it's doing it to see what's actually going on. This is very similar to [ARM](https://learn.microsoft.com/en-us/azure/azure-resource-manager/management/overview) in Azure in that they both provide infrastructure as code (IaC) capabilities for managing cloud resources. After this is done, you should be able to see the data in your s3 bucket by going to workspace homepage on Databricks -> Click on `+ New` on top left -> `Add or upload data` -> `Create table from Amazon S3`, and under 
`Select all` you will see the data in the external s3 location.

## Task 2: Read data from the S3 bucket to Databricks

In [0]:
# Read all partitions for all three topics
df_pin = spark.read.json("s3a://pinterest-confluent-kafka-connect-s3/topics/808492447622.pin/partition=*/")
df_geo = spark.read.json("s3a://pinterest-confluent-kafka-connect-s3/topics/808492447622.geo/partition=*/")
df_user = spark.read.json("s3a://pinterest-confluent-kafka-connect-s3/topics/808492447622.user/partition=*/")

# Verify the combined data
# df_pin.show(10)
display(df_pin.head(5))
display(df_geo.head(5))
display(df_user.head(5))

In [0]:
# Sum up all the values in the "downloaded" column
from pyspark.sql.functions import col, sum
total_sum = df_pin.select(sum(col("downloaded"))).collect()[0][0]

# Print the total sum
print(f"The total sum of the 'downloaded' column is: {total_sum}")
# The result might show not all downloaded values are 1.

In [0]:
# Print the number of rows
num_rows = df_pin.count()
num_rows = df_geo.count()
num_rows = df_user.count()

print(f'The number of rows in the df_pin is: {num_rows}')
print(f'The number of rows in the df_geo is: {num_rows}')
print(f'The number of rows in the df_user is: {num_rows}')

# Print the schema of the three DataFrames
print("Schema for df_pin:")
df_pin.printSchema()
print("Schema for df_geo:")
df_geo.printSchema()
print("Schema for df_user:")
df_user.printSchema()

# Milestone 6: Batch Processing: Spark on Databricks

## Task 1: Clean the DataFrame that contains information about Pinterest posts.

In [0]:
from pyspark.sql.functions import col, trim, when, regexp_replace

# Replace empty entries and entries with no relevant data in string columns with None
string_columns = [col_name for col_name, dtype in df_pin.dtypes if dtype == 'string']
for column in string_columns:
    df_pin = df_pin.withColumn(
        column, 
        when(trim(col(column)) != "", col(column)).otherwise(None)
    )
# Replace invalid or missing entries in numeric columns with None
numeric_columns = [col_name for col_name, dtype in df_pin.dtypes if dtype in ['int', 'double', 'long']]
for column in numeric_columns:
    df_pin = df_pin.withColumn(
        column, 
        when(col(column).isNotNull(), col(column)).otherwise(None)
    )

# Transform follower_count to integer with multiple suffix cases.
df_pin = df_pin.withColumn(
    "follower_count",
    # Regex replacements to handle suffixes like k (thousand), M (million), B (billion). (?i)" makes the pattern case-insensitive while '$' anchors the match to the end of the string.
    regexp_replace(
        regexp_replace(
            regexp_replace(
                col("follower_count"), 
                "(?i)k$", "000"                                          # 1k -> 1000
            ),
            "(?i)m$", "000000"                                           # 1M -> 1000000
        ),
        "(?i)b$", "000000000"                                            # 1B -> 1000000000
    )
)

# Ensure numeric columns are correctly typed
numeric_columns = ["downloaded", "follower_count", "index"]
for column in numeric_columns:
    df_pin = df_pin.withColumn(
        column,
        col(column).cast("int")
    )

# Clean save_location column to keep only the path by removing the prefix "Local save in " from its values
df_pin = df_pin.withColumn(
    "save_location",
    regexp_replace(col("save_location"), "^Local save in ", "") # '^' anchors the match to the start of the string.
)

# Rename index column to ind
df_pin = df_pin.withColumnRenamed("index", "ind")

# Reorder columns
# desired_order = [
#     "ind", "unique_id", "title", "description", "follower_count",
#     "downloaded", "poster_name", "tag_list", "is_image_or_video", 
#     "image_src", "save_location", "category"
# ]
desired_order = [
    "ind", "unique_id", "title", "description", "follower_count", 
    "poster_name", "tag_list", "is_image_or_video", 
    "image_src", "save_location", "category"
]
df_pin = df_pin.select(desired_order)

In [0]:
# Display the first 5 rows of the cleaned DataFrame `df_pin` to inspect the data
display(df_pin.head(5))

# Count the total number of rows in the cleaned DataFrame `df_pin`
num_rows = df_pin.count()

# Print the total number of rows after cleaning the DataFrame
print(f'The number of rows after cleaning df_pin in Task 1 is: {num_rows}')

# Print the schema of the cleaned DataFrame `df_pin` to verify column names and data types
df_pin.printSchema()

## Task 2: Clean the DataFrame that contains information about geolocation.

In [0]:
from pyspark.sql.functions import array, col, to_timestamp

# Create a new column "coordinates" as an array of latitude and longitude
df_geo = df_geo.withColumn("coordinates", array(col("latitude"), col("longitude")))

# Drop the latitude and longitude columns
df_geo = df_geo.drop("latitude", "longitude")

# Convert the "timestamp" column from string to timestamp
df_geo = df_geo.withColumn("timestamp", to_timestamp(col("timestamp")))

# Reorder the columns
df_geo = df_geo.select("ind", "country", "coordinates", "timestamp")

# Show the cleaned DataFrame schema
print("Schema for df_geo:")
df_geo.printSchema()

## Task 3: Clean the DataFrame that contains information about users.

In [0]:
from pyspark.sql.functions import concat_ws, col, to_timestamp

# Create a new column "user_name" by concatenating "first_name" and "last_name"
df_user = df_user.withColumn("user_name", concat_ws(" ", col("first_name"), col("last_name")))

# Drop the "first_name" and "last_name" columns
df_user = df_user.drop("first_name", "last_name")

# Convert the "date_joined" column from string to timestamp
df_user = df_user.withColumn("date_joined", to_timestamp(col("date_joined")))

# Reorder the columns
df_user = df_user.select("ind", "user_name", "age", "date_joined")

# Show the cleaned DataFrame schema
print("Schema for df_user:")
df_user.printSchema()

# Print out the first 5 rows to verify the data (particularly the "user_name" column)
display(df_user.head(5))

## Task 4: Find the most popular category in each country.